In [1]:
import os
import pandas as pd
import numpy as np
import networkx as nx
from tqdm.auto import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
DATA_DIR = "/kaggle/input/cafa-6-protein-function-prediction"
OBO_PATH = os.path.join(DATA_DIR, "Train/go-basic.obo")
IA_PATH = os.path.join(DATA_DIR, "IA.tsv")

# YOUR INPUT FILE
SUBMISSION_INPUT = '/kaggle/input/cafa6-protein-function-enhanced-nb-v2/submission.tsv'
SUBMISSION_OUTPUT = 'submission.tsv'

# ==========================================
# 1. LOAD IA WEIGHTS & GRAPH
# ==========================================
def load_resources(obo_path, ia_path):
    print("Loading IA Weights and GO Hierarchy...")
    # IA weights are the secret to moving from 0.38 to 0.46
    ia_df = pd.read_csv(ia_path, sep='\t', header=None, names=['term', 'ia'])
    ia_map = dict(zip(ia_df['term'], ia_df['ia']))
    
    graph = nx.DiGraph()
    with open(obo_path, "r") as f:
        cur_id = None
        for line in f:
            if line.startswith("id: "):
                cur_id = line.split("id: ")[1].strip()
                graph.add_node(cur_id)
            elif line.startswith("is_a: ") and cur_id:
                parent = line.split("is_a: ")[1].split(" ! ")[0].strip()
                graph.add_edge(cur_id, parent)
                
    topo_order = list(nx.topological_sort(graph))
    parents_map = {node: list(graph.successors(node)) for node in graph.nodes()}
    return topo_order, parents_map, ia_map

# ==========================================
# 2. IA-OPTIMIZED PROPAGATION
# ==========================================
def optimized_propagation(df, topo_order, parents_map, ia_map):
    print("Applying Information Accretion Weighting...")
    
    # Filter: Roots and generic terms (IA=0) are removed to boost Precision
    valid_terms = {term for term, ia in ia_map.items() if ia > 0}
    
    results = []
    for pid, group in tqdm(df.groupby('protein_id')):
        scores = dict(zip(group['go_term'], group['score']))
        
        # True Path Rule: Ensure parent score >= child score
        for child in topo_order:
            if child in scores:
                c_score = scores[child]
                for parent in parents_map.get(child, []):
                    scores[parent] = max(scores.get(parent, 0), c_score)
        
        # SELECTION STRATEGY FOR 0.46+ RANK:
        # We calculate a 'Value Score' using IA weights
        value_list = []
        for term, score in scores.items():
            if term in valid_terms and score >= 0.05:
                ia_val = ia_map.get(term, 0.0)
                # IA-Boost: Prioritize specific terms that give more points
                rank_score = score * (1 + (ia_val / 10.0))
                value_list.append((pid, term, round(score, 3), rank_score))
        
        # Keep top 150 most valuable terms per protein to avoid noise penalty
        value_list.sort(key=lambda x: x[3], reverse=True)
        for val in value_list[:150]:
            results.append(val[:3])
                
    return pd.DataFrame(results, columns=['protein_id', 'go_term', 'score'])

# ==========================================
# 3. EXECUTION
# ==========================================
topo, parents, ia = load_resources(OBO_PATH, IA_PATH)

# Robust loading for submission files
sub = pd.read_csv(SUBMISSION_INPUT, sep='\t', header=None, 
                 names=['protein_id', 'go_term', 'score'], 
                 on_bad_lines='skip', engine='python')

final_sub = optimized_propagation(sub, topo, parents, ia)
final_sub.to_csv(SUBMISSION_OUTPUT, sep='\t', index=False, header=False)
print(f"Optimization complete! Saved {len(final_sub)} high-value predictions.")

Loading IA Weights and GO Hierarchy...
Applying Information Accretion Weighting...


  0%|          | 0/279437 [00:00<?, ?it/s]

Optimization complete! Saved 27820767 high-value predictions.
